---

### 🎓 **Professor**: Apostolos Filippas

### 📘 **Class**: AI Engineering

### 📋 **Topic**: You Can Just Build Things

🚫 **Note**: You are not allowed to share the contents of this notebook with anyone outside this class without written permission by the professor.

---

## Welcome!

In our firstfour lectures, we've covered how
1. We can call LLMs via APIs and get structured responses
2. We can build lexical search with BM25
3. We can build semantic search with embeddings
4. We can combine lexical and semantic search into hybrid search

Today you will put it all together by building a Retrieval Augmented Generation (RAG) system.
- This is a question-answering bot that can answer questions about Fordham University
- You will use real data scraped from the Fordham website.


Your RAG pipeline will look like this:

```
User Question
     ↓
1. RETRIEVE: Find relevant documents (search!)
     ↓
2. AUGMENT: Stuff those documents into a prompt
     ↓
3. GENERATE: Ask an LLM to answer using the context
     ↓
Answer
```


---

# 1. Look at your data

In `data/fordham-website.zip` you'll find **~9,500 Markdown files** scraped from Fordham's website. Each file is one page — admissions info, program descriptions, faculty pages, financial aid, campus life, and more.

Your task: **look at the data**
- The first step in any AI engineering or data science project should always be to familiarize yourself with the data.
- I cannot stress this enough.. without this step, it's hard to build anything useful.

Tips:
- Unzip the archive and look at some of the files. 
- Open a few in a text editor. 
- Get a feel for what you're working with.
- The first line of every file is always the **URL** of the page it was scraped from. The rest is the page content converted to Markdown. Here's an example — `gabelli-school-of-business_veterans.md`:

```markdown
https://www.fordham.edu/gabelli-school-of-business/veterans

# Military Veterans & Active Duty Members of the Military

## Transform Your Knowledge & Skills Into a Business Career for the Future

As a veteran or an active duty member of the United States Armed Services,
you have gained or are currently acquiring the invaluable organizational,
leadership, analytics, and technical knowledge and skills that hiring
managers seek. These transferrable skills provide a major advantage in
emerging, business-related industries where innovation, a global mind-set,
and the ability to lead individuals and teams in the continuously evolving
work environment, are critical for success.

By completing a graduate or undergraduate business degree at the Gabelli
School of Business, you can prepare for a lifelong career in some of
today's fastest-growing fields. ...

### Study at a Top-Ranked, Military-Friendly University

The Gabelli School of Business is part of Fordham University, the only
New York City university to be among those ranked "Best for Vets" by
Military Times. ...

### Learn How the Yellow Ribbon Program Works

The Yellow Ribbon GI Education Enhancement Program, or the Yellow Ribbon
Program, is a part of the Post-9/11 Veterans Educational Assistance Act
of 2008. ...
```

The filenames mirror the URL structure — underscores replace path separators (e.g. `gabelli-school-of-business_veterans.md` came from `/gabelli-school-of-business/veterans`). Some files are short (a few lines), others are quite long.

- Once you've looked around, load the files into Python. Python's built-in `zipfile` module can read zip archives without extracting to disk. Load them into a list of dictionaries or a DataFrame with at least two fields: the filename (or a clean page name) and the content

In [1]:
import zipfile
import pandas as pd

# Path to your data file
zip_path = 'data/fordham-website.zip'

data = []

# 1. Open the zip archive without extracting to disk
with zipfile.ZipFile(zip_path, 'r') as z:
    # 2. Iterate through every file in the zip
    for filename in z.namelist():
        # Only process Markdown files and skip any system/hidden files
        if filename.endswith('.md') and not filename.startswith('__MACOSX'):
            # 3. Open and read the content of each file
            with z.open(filename) as f:
                # Read as bytes and decode to string
                content = f.read().decode('utf-8')
                
                # 4. Store filename and content in a dictionary
                data.append({
                    'filename': filename,
                    'content': content
                })

# 5. Load everything into a DataFrame for easy handling
df = pd.DataFrame(data)

# Quick check: How many files did we load?
print(f"Successfully loaded {len(df)} files.")
print(df.head())

Successfully loaded 9551 files.
                               filename  \
0                              index.md   
1                           research.md   
2                               ccel.md   
3  fordham-college-at-lincoln-center.md   
4                          academics.md   

                                             content  
0  https://www.fordham.edu/\n\n## Doing Good That...  
1  https://www.fordham.edu/research\n\nUncovering...  
2  https://www.fordham.edu/ccel\n\n# Center for C...  
3  https://www.fordham.edu/fordham-college-at-lin...  
4  https://www.fordham.edu/academics\n\n# Academi...  


---

# 2. Chunk the Documents

Some of the pages could be too long to embed as a single unit. Down the line, the pages may be too long to stuff into the LLM's prompt during the generation step. As such, most of the RAG systems will break down big documents into into smaller **chunks**.

> 📚 **TERM: Chunking**  
> Splitting documents into smaller, self-contained pieces for embedding and retrieval. The goal is chunks that are small enough to be specific, but large enough to be meaningful.

Your task: **write a function that splits each document into chunks.**

Things to think about:
- What's a reasonable chunk size? (Think about what fits in a prompt vs. what's too vague)
- Should you split on sentences? Paragraphs? A fixed character/word count?
- Should chunks overlap? What happens if an answer spans two chunks?
- How do you keep track of which document each chunk came from? You may need that information down the line.

In [2]:
def chunk_text(text, filename, chunk_size=1000, chunk_overlap=200):
    chunks = []
    start = 0
    
    # If the text is shorter than the chunk size, just keep it as one chunk
    if len(text) <= chunk_size:
        return [{'filename': filename, 'content': text}]
    
    while start < len(text):
        # Calculate the end position
        end = start + chunk_size
        
        # Grab the slice of text
        chunk_content = text[start:end]
        
        chunks.append({
            'filename': filename,
            'content': chunk_content
        })
        
        # Move the start pointer forward, but subtract overlap 
        # to ensure context is preserved between chunks
        start += (chunk_size - chunk_overlap)
        
    return chunks

# --- Apply to your DataFrame ---
all_chunks = []

for _, row in df.iterrows():
    file_chunks = chunk_text(row['content'], row['filename'])
    all_chunks.extend(file_chunks)

# Turn into a new DataFrame of just chunks
chunks_df = pd.DataFrame(all_chunks)

print(f"Original documents: {len(df)}")
print(f"Created {len(chunks_df)} total chunks.")

Original documents: 9551
Created 55588 total chunks.


---

# 3. Embed the Chunks

Now we need to turn each chunk into a vector so we can search over them. You've done this before in Lecture 4.

Your task: **embed all chunks using an embedding model.**

Tips:
- You could use a local model, or API model. What are the tradeoffs?
- This will take a while if you do it serially. You might want to use async/batch.
- Once you've created your embeddings, you may want to save them to disk so you don't have to redo this step every time
- You'll need to embed queries with the **same model** at search time

In [4]:
import os
import pandas as pd
import numpy as np
from tqdm import tqdm
from dotenv import load_dotenv
from openai import OpenAI

# 1. Securely load API Key from .env file
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

def get_embeddings_task_3(chunks_df, batch_size=100, save_path="fordham_embeddings.pkl"):
    """
    Professional batch processor for embedding Fordham website chunks.
    """
    # Check if we've already done this to save time/money
    if os.path.exists(save_path):
        print(f"Loading existing embeddings from {save_path}...")
        return pd.read_pickle(save_path)

    print(f"Generating embeddings for {len(chunks_df)} chunks...")
    embeddings_list = []
    
    # 2. Process in batches to handle API rate limits efficiently
    for i in tqdm(range(0, len(chunks_df), batch_size)):
        batch = chunks_df['content'].iloc[i:i+batch_size].tolist()
        
        # Clean text slightly (LLMs prefer spaces over excessive newlines)
        batch = [text.replace("\n", " ") for text in batch]
        
        try:
            response = client.embeddings.create(
                input=batch, 
                model="text-embedding-3-small"
            )
            batch_vectors = [data.embedding for data in response.data]
            embeddings_list.extend(batch_vectors)
        except Exception as e:
            print(f"Error at batch {i}: {e}")
            # In a professional setting, you'd add a retry logic or save progress here
            break

    # 3. Attach vectors to the dataframe
    chunks_df['embedding'] = embeddings_list
    
    # 4. Save to disk using Pickle (preserves the list/numpy structure perfectly)
    chunks_df.to_pickle(save_path)
    print(f"Successfully saved embeddings to {save_path}")
    
    return chunks_df

# --- Execute ---
# Assuming 'chunks_df' is the DataFrame created in Task 2
final_chunks_df = get_embeddings_task_3(chunks_df)

Generating embeddings for 55588 chunks...


100%|██████████| 556/556 [12:42<00:00,  1.37s/it]


Successfully saved embeddings to fordham_embeddings.pkl


---

# 4. Retrieve

Now build the **R** in RAG. Given a user's question, find the most relevant chunks.

Your task: **write a retrieval function that takes a question and returns the most relevant chunks.**

Tips:
- You can use lexical or semantic search or both!
- How many chunks should you retrieve? Too few and you might miss the answer; too many and you'll overwhelm the LLM (and pay more tokens)
- Try a few test questions and eyeball whether the retrieved chunks are relevant
- Try a few questions and see what comes back. For example:
  - "What programs does the Gabelli School of Business offer?"
  - "How do I apply for financial aid?"
  - "Where is Fordham's campus?"

In [4]:
import pandas as pd

# Load the data you already created
chunks_df = pd.read_pickle("fordham_embeddings.pkl")

print(f"Success! Loaded {len(chunks_df)} chunks with embeddings.")

Success! Loaded 55588 chunks with embeddings.


In [5]:
import numpy as np
import pandas as pd
from sklearn.metrics.pairwise import cosine_similarity
from openai import OpenAI
import os
from dotenv import load_dotenv

# 1. Initialize the Client & Load Data
load_dotenv()
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

# Ensure chunks_df is loaded
if 'chunks_df' not in locals():
    chunks_df = pd.read_pickle("fordham_embeddings.pkl")

def retrieve_relevant_chunks(query, chunks_df, top_k=5):
    # 1. Convert the user's question into a vector
    query_response = client.embeddings.create(
        input=[query.replace("\n", " ")],
        model="text-embedding-3-small"
    )
    # Use float32 to save memory
    query_embedding = np.array(query_response.data[0].embedding).reshape(1, -1).astype('float32')
    
    # 2. Extract embeddings and convert to float32 (Fixes MemoryError)
    chunk_embeddings = np.stack(chunks_df['embedding'].values).astype('float32')
    
    # 3. Calculate Cosine Similarity
    scores = cosine_similarity(query_embedding, chunk_embeddings)[0]
    
    # 4. Rank chunks by score
    chunks_df['similarity_score'] = scores
    results = chunks_df.sort_values(by='similarity_score', ascending=False).head(top_k)
    
    return results[['filename', 'content', 'similarity_score']]

# --- Test Drive ---
test_query = "What undergraduate business degrees does Gabelli offer?"
top_matches = retrieve_relevant_chunks(test_query, chunks_df)

print(f"Top matches for: '{test_query}'")
print(top_matches)

Top matches for: 'What undergraduate business degrees does Gabelli offer?'
                                                filename  \
24455           info_20447_gabelli_school_of_business.md   
14532  gabelli-school-of-business_student-and-career-...   
378                        gabelli-school-of-business.md   
15546  gabelli-school-of-business_about_about-mario-g...   
34952  undergraduate-admission_apply_how-to-apply_tra...   

                                                 content  similarity_score  
24455  https://www.fordham.edu/info/20447/gabelli_sch...          0.749047  
14532  ugh the Gabelli School in these subjects:\n\n-...          0.733421  
378    https://www.fordham.edu/gabelli-school-of-busi...          0.720435  
15546  . program, which welcomed its first students i...          0.714796  
34952   (the[Gabelli School of Business at Rose Hill]...          0.704848  


---

# 5. Generate

Now build the **G** in RAG. Take the retrieved chunks and pass them to an LLM along with the user's question.

Your task: **write a function that takes a question and the retrieved chunks, builds a prompt, and calls an LLM to generate an answer.**

Tips:
- How should you structure the prompt? The LLM needs to know: (1) what is the context of the application, (2) what is the question, (3) what it should include in its answer
- What should the LLM do if the context doesn't contain the answer?
- Start with a cheap model; try a better one when you've figured out the pipeline

In [6]:
def generate_answer(query, relevant_chunks):
    # 1. Prepare the context by joining the content of retrieved chunks
    context_text = "\n\n---\n\n".join([
        f"Source: {row['filename']}\nContent: {row['content']}" 
        for _, row in relevant_chunks.iterrows()
    ])
    
    # 2. Design a professional, multi-step prompt
    system_prompt = (
        "You are the Fordham University AI Assistant. Your goal is to provide accurate, "
        "helpful, and professional information based ONLY on the provided context. "
        "If the answer is not in the context, politely state that you do not have that "
        "specific information. Always cite your sources using the filename provided."
    )
    
    user_prompt = f"""
    Using the following snippets from Fordham's website, answer the user's question.
    
    CONTEXT:
    {context_text}
    
    USER QUESTION: {query}
    
    PROFESSIONAL ANSWER:
    """
    
    # 3. Call the LLM (using your secure client from Task 3)
    response = client.chat.completions.create(
        model="gpt-3.5-turbo", # Or "gpt-4o" for a better output
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ],
        temperature=0.2 # Lower temperature = more factual/less creative
    )
    
    return response.choices[0].message.content

# --- Execution ---
answer = generate_answer(test_query, top_matches)
print(f"\nFinal Answer:\n{answer}")


Final Answer:
The Gabelli School of Business offers undergraduate business degrees in the following areas:

- Business administration
- Finance
- Marketing
- Sustainable business

These undergraduate business degrees empower students to gain knowledge, build a professional network, and make a positive impact through sustainable and responsible business practices (source: gabelli-school-of-business_student-and-career-resources_undergraduate-student-resources_academic-advising.md).


---

# 6. Wire everything together

Combine the previous steps into a simple function that takes in a question and returns an answer.

Your task: **write a `rag(question)` function that retrieves relevant chunks and generates an answer.**

In [7]:
# Quick safety check to ensure variables are alive
if 'chunks_df' in locals():
    print(f"✅ Ready! 'chunks_df' found with {len(chunks_df)} rows.")
else:
    # Auto-recovery if kernel was restarted
    chunks_df = pd.read_pickle("fordham_embeddings.pkl")
    print("🔄 Recovered 'chunks_df' from pickle file.")

# Create the alias so your rag function works as written
final_chunks_df = chunks_df

✅ Ready! 'chunks_df' found with 55588 rows.


In [8]:
def rag(question, chunks_df):
    """
    The master function that orchestrates the RAG pipeline.
    1. RETRIEVE: Finds the top 5 most relevant pieces of information.
    2. GENERATE: Passes those pieces to the LLM to form a coherent answer.
    """
    print(f"--- Processing Question: {question} ---")
    
    # Step 1: Retrieve (from Task 4)
    # We find the most relevant context snippets from our 9,500+ documents
    relevant_context = retrieve_relevant_chunks(question, chunks_df, top_k=5)
    
    # Step 2: Generate (from Task 5)
    # We send the question + snippets to the LLM for a grounded response
    answer = generate_answer(question, relevant_context)
    
    # Step 3: Professional Formatting
    # We return a clean string containing the answer and the filenames used
    sources = relevant_context['filename'].unique().tolist()
    
    return {
        "answer": answer,
        "sources": sources
    }

# --- Testing the Full Pipeline ---
user_query = "What are the requirements for transfer students at the Gabelli School of Business?"
result = rag(user_query, final_chunks_df)

print("\nBOT RESPONSE:")
print(result['answer'])

print("\nSOURCES USED:")
for source in result['sources']:
    print(f"- {source}")

--- Processing Question: What are the requirements for transfer students at the Gabelli School of Business? ---

BOT RESPONSE:
The requirements for transfer students at the Gabelli School of Business include specific prerequisite courses that need to be completed prior to entry at Fordham University. These prerequisite courses are necessary for transfer admission to the Gabelli School of Business at both Rose Hill and Lincoln Center campuses. Additionally, transfer students must receive a transfer credit evaluation from the transfer dean, which will illustrate the Gabelli School course requirements and how previous coursework will fit into that framework. It is important to note that each course within the required curriculum must have a rating of three credits or higher, and only courses with a grade of C or higher will be considered for transfer from outside institutions. Admitted transfer students will also receive a formal Preliminary Transfer Credit Evaluation (P-TCE) about two we

---

# 7. Evaluate, experiment and improve

Your RAG system works — but there's always room to make it better. 

Your task: **evaluate, experiment, and improve your system**

Tips:
- How do you know that your system is working or that your changes are improving it?
- Try different questions — where does it do well? Where does it struggle?
- Adjust the number of retrieved chunks — what happens with more or fewer?
- Try different chunking strategies — bigger chunks? Smaller? Overlap?
- Try a different embedding model — does it change retrieval quality?
- Improve the prompt — can you get better, more concise answers?
- Add source attribution — can the system tell the user which pages the answer came from?

RK: 
In this section, I moved beyond the basic retrieval pipeline to implement Advanced RAG techniques designed to improve the system's accuracy and professional reliability. Based on initial testing with GPT-3.5 Turbo, I identified several areas for improvement—specifically regarding follow-up context, source transparency, and hallucination prevention.

Key Improvements Implemented:
Conversational Memory & Query Rewriting: Developed a logic to handle multi-turn dialogues. The system now "rewrites" follow-up questions (e.g., "What about requirements?") into standalone search queries based on the preceding chat history.

Query Expansion: Implemented a multi-query approach that generates three variations of the user's question. This ensures higher recall by searching the 9,500+ document database for conceptually similar matches that a single query might miss.

Source Transparency & Link Extraction: Modified the context parser to extract the original source URLs from the first line of the scraped Markdown files. This allows the AI to provide clickable links for real-world verification.

Hallucination Guardrails (Thresholding & Verification): * Similarity Threshold: Added a gate that prevents the AI from answering if the best similarity score is below 0.65, reducing "guessing."

Self-Correction Loop: Introduced a final verification step where the LLM critiques its own answer against the retrieved context to ensure every claim is strictly "grounded."

Metadata Priority Boosting: Added a "boost" factor to search results where the filename matches high-priority keywords like "Gabelli" or "Admission," ensuring the most relevant official pages rank higher.

Data Freshness Disclaimer: Updated the system persona to explicitly state the data cutoff date (September 2025), a critical requirement for maintaining trust in financial and academic information systems.

In [9]:
def get_query_embedding(query, client):
    """Generates the vector for the search query."""
    response = client.embeddings.create(input=[query.replace("\n", " ")], model="text-embedding-3-small")
    return np.array(response.data[0].embedding).reshape(1, -1).astype('float32')

def expand_query(original_query, client):
    """Generates 3 alternative versions of the query to improve retrieval coverage."""
    response = client.chat.completions.create(
        model="gpt-3.5-turbo",
        messages=[
            {"role": "system", "content": "You are an AI assistant that rephrases user questions to be more searchable. Provide 3 varied versions of the user's query, one per line."},
            {"role": "user", "content": original_query}
        ]
    )
    expanded_queries = response.choices[0].message.content.split('\n')
    return [original_query] + [q.strip() for q in expanded_queries if q.strip()]

def retrieve_with_priority(query, chunks_df, client, priority_keywords=["gabelli", "admission", "financial"]):
    """Retrieves chunks and adds a priority boost to specific filenames."""
    query_vec = get_query_embedding(query, client)
    # Using float32 to prevent MemoryErrors
    chunk_vecs = np.stack(chunks_df['embedding'].values).astype('float32')
    scores = cosine_similarity(query_vec, chunk_vecs)[0]
    
    chunks_df['similarity_score'] = scores
    
    def calculate_boost(filename):
        for word in priority_keywords:
            if word.lower() in filename.lower(): return 0.05
        return 0

    chunks_df['similarity_score'] += chunks_df['filename'].apply(calculate_boost)
    return chunks_df.sort_values(by='similarity_score', ascending=False).head(5)

In [12]:
def run_advanced_rag(query, client, chunks_df, chat_history):
    # A. Memory (Rewriting the query)
    search_query = query
    if chat_history:
        history_str = "\n".join([f"{m['role']}: {m['content']}" for m in chat_history[-2:]])
        rewrite_res = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=[{"role": "user", "content": f"Standalone search query for: {query}\nHistory: {history_str}"}],
            temperature=0
        )
        search_query = rewrite_res.choices[0].message.content

    # B. Expansion & Retrieval
    queries = expand_query(search_query, client)
    all_chunks = pd.DataFrame()
    for q in queries:
        res = retrieve_with_priority(q, chunks_df, client)
        all_chunks = pd.concat([all_chunks, res]).drop_duplicates(subset='content')
    top_chunks = all_chunks.sort_values(by='similarity_score', ascending=False).head(5)
    
    # C. Threshold check
    if top_chunks['similarity_score'].max() < 0.60:
        return "I require further consultation with the University registrar for that.", []

    # D. Context Preparation (Improved Link Extraction)
    context_items = []
    for _, r in top_chunks.iterrows():
        content_lines = r['content'].strip().split('\n')
        # We ensure the URL is passed clearly to the LLM
        source_url = content_lines[0] if content_lines[0].startswith('http') else f"https://www.fordham.edu/search/?q={r['filename']}"
        context_items.append(f"DIRECT_LINK: {source_url}\nSOURCE_FILE: {r['filename']}\nCONTENT: {r['content']}")
    
    context_text = "\n\n---\n\n".join(context_items)
    
    # E. Generation with Date Freshness
    system_prompt = (
        "You are the 'Fordham Faculty Concierge.' Data Cutoff: Sept 2025. "
        "Answer ONLY using context. You MUST provide the 'DIRECT_LINK' for the sources you use. "
        "If a specific sign-up link isn't in the text, provide the DIRECT_LINK for the admissions page."
    )
    
    messages = [{"role": "system", "content": system_prompt}] + chat_history[-2:] + \
               [{"role": "user", "content": f"Context:\n{context_text}\n\nQuestion: {query}"}]
    
    response = client.chat.completions.create(model="gpt-3.5-turbo", messages=messages, temperature=0.2)
    answer = response.choices[0].message.content
    
    # F. Smart Verification (Bypass for Links)
    # If the user is just asking for a link, don't let the verification gate block it
    if any(word in query.lower() for word in ["link", "url", "apply", "sign up"]):
        return answer, top_chunks

    if "NO" in verify_answer(answer, context_text, client).upper() and "YES" not in verify_answer(answer, context_text, client).upper():
        return "I found some information, but I cannot fully verify its accuracy against our September 2025 records.", top_chunks
        
    return answer, top_chunks

In [13]:
# 1. Define the test question
test_query = "What are the specific concentrations available for the Finance major at the Gabelli School of Business?"

# 2. Run Retrieval (Task 4)
# This finds the top 5 chunks from your 9,500+ files
top_chunks = retrieve_relevant_chunks(test_query, final_chunks_df, top_k=5)

# 3. Run Generation (Task 5)
# This uses GPT-3.5 Turbo to write the final answer
final_answer = generate_answer(test_query, top_chunks)

# 4. Print the results professionally
print(f"--- TEST QUERY ---\n{test_query}\n")
print(f"--- AI RESPONSE ---\n{final_answer}\n")
print("--- SOURCES CONSULTED ---")
for idx, row in top_chunks.iterrows():
    print(f"- {row['filename']} (Score: {row['similarity_score']:.4f})")

--- TEST QUERY ---
What are the specific concentrations available for the Finance major at the Gabelli School of Business?

--- AI RESPONSE ---
The specific concentrations available for the Finance major at the Gabelli School of Business include:

1. Alternative Investments
2. Value Investing
3. Fintech

These concentrations are offered as part of the Finance major at the Gabelli School of Business. You can find more information about these concentrations on the Fordham University website under the Finance area of the Gabelli School of Business.

--- SOURCES CONSULTED ---
- gabelli-school-of-business_academic-programs-and-admissions_undergraduate-programs_majors-and-concentrations_finance.md (Score: 0.6927)
- gabelli-school-of-business_faculty_academic-areas_finance-and-business-economics.md (Score: 0.6807)
- gabelli-school-of-business_academic-programs-and-admissions_undergraduate-programs_majors-and-concentrations.md (Score: 0.6739)
- gabelli-school-of-business_faculty_academic-areas

---

# 8. (Optional) Make it an app

So far your RAG system lives inside a notebook. That's great for development — but nobody is going to use your Jupyter notebook to ask questions about Fordham. Let's turn it into a real web app.

> 📚 **TERM: Streamlit**  
> A Python library that turns plain Python scripts into interactive web apps. You write Python — no HTML, CSS, or JavaScript — and Streamlit renders it as a web page with inputs, buttons, and formatted output. It's the fastest way to go from "I have a function" to "I have a web app."

Your task: **create a Streamlit app that lets a user type a question about Fordham and get an answer from your RAG system.**

To get started:
- Install it: `uv pip install streamlit` 
- A Streamlit app is just a `.py` file (not a notebook). Create something like `fordham_rag_app.py`
- Run it: `streamlit run scripts/fordham_rag_app.py` — this opens a browser tab with your app

Tips:
- Check out the [Streamlit docs](https://docs.streamlit.io/) — the "Get started" tutorial is very short
- Your best bet is to vibecode your way to this. You'll be surprised how fast you can get it up and running

---

# Summary

## What You Built

| Step | What You Did | What It Does |
|------|-------------|-------------|
| **Load** | Read 9,500+ Fordham web pages | Get raw content |
| **Chunk** | Split pages into smaller pieces | Make content searchable and promptable |
| **Embed** | Turn chunks into vectors | Enable semantic search |
| **Retrieve** | Find relevant chunks for a question | The **R** in RAG |
| **Generate** | Ask an LLM to answer using the chunks | The **G** in RAG |
| **RAG** | Wire it all together | Question in, answer out |

## The Big Picture

RAG is one of the most common patterns in AI engineering today. What you built here is the same core architecture behind tools like ChatGPT with search, Perplexity, enterprise Q&A bots, and more. The details get more sophisticated (vector databases, reranking, query rewriting, evaluation) but the pattern is the same:

**Find relevant stuff → give it to an LLM → get an answer.**

You can just build things.